# Generate test records in bronze (CDC flow validation)

Inserts a small, controlled set of **new** records into `claudecatalog.bronze` (airports, passengers, bookings) to validate the incremental CDC flow of the `flight_data_prep_silver_pipeline` Lakeflow pipeline.

- Writes are **idempotent** (`MERGE` on primary keys, insert-only) — safe to re-run.
- Keys: airports `A9001/A9002`, passengers `P9001/P9002/P9003`, bookings `B95001…B95008`.

After running this notebook, trigger a pipeline update (or let the scheduled job run) and then run `02_validate_cdc_flow`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

AIRPORTS_SCHEMA = StructType([
    StructField("airport_id", StringType(), False),
    StructField("airport_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
])

PASSENGERS_SCHEMA = StructType([
    StructField("passenger_id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("nationality", StringType(), True),
])

BOOKINGS_SCHEMA = StructType([
    StructField("booking_id", StringType(), False),
    StructField("passenger_id", StringType(), True),
    StructField("flight_id", StringType(), True),
    StructField("airport_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("booking_date", StringType(), True),  # bronze stores yyyy-MM-dd strings
])

test_airports = spark.createDataFrame([
    ("A9001", "Nimbusport International Airport", "Cloud City", "Fictionland"),
    ("A9002", "Test Valley International Airport", "Testville", "Testland"),
], AIRPORTS_SCHEMA)

test_passengers = spark.createDataFrame([
    ("P9001", "Alice Test", "Female", "Fictionland"),
    ("P9002", "Bob Test", "Male", "Testland"),
    ("P9003", "Carol Test", "Female", "Narnia"),
], PASSENGERS_SCHEMA)

# Mix of new (P900x/A900x) and existing (P0001/A001) entities to exercise both paths
test_bookings = spark.createDataFrame([
    ("B95001", "P9001", "F0100", "A9001", 750.00, "2025-07-01"),
    ("B95002", "P9001", "F0101", "A9002", 125.50, "2025-07-01"),
    ("B95003", "P9002", "F0102", "A9001", 480.25, "2025-07-02"),
    ("B95004", "P9002", "F0103", "A001",  910.75, "2025-07-02"),
    ("B95005", "P9003", "F0104", "A9002", 320.10, "2025-07-03"),
    ("B95006", "P9003", "F0105", "A001",  640.60, "2025-07-03"),
    ("B95007", "P0001", "F0106", "A9001", 275.40, "2025-07-04"),
    ("B95008", "P0001", "F0107", "A001",  815.85, "2025-07-04"),
], BOOKINGS_SCHEMA)

print("Test records defined:", test_airports.count(), "airports,",
      test_passengers.count(), "passengers,", test_bookings.count(), "bookings")

In [0]:
# Input validation: structured errors, never a raw crash
results = []

def validate(df, key_col, label):
    """Check key integrity; record a structured result."""
    try:
        null_keys = int(df.filter(F.col(key_col).isNull()).count())
        dup_keys = int(df.groupBy(key_col).count().filter(F.col("count") > 1).count())
        results.append((label, key_col, null_keys, dup_keys,
                        "OK" if null_keys == 0 and dup_keys == 0 else "INVALID_INPUT"))
    except Exception as e:
        results.append((label, key_col, None, None, f"VALIDATION_ERROR: {type(e).__name__}: {e}"))

validate(test_airports, "airport_id", "airports")
validate(test_passengers, "passenger_id", "passengers")
validate(test_bookings, "booking_id", "bookings")

validation_df = spark.createDataFrame(
    results, schema="entity STRING, key_column STRING, null_keys INT, duplicate_keys INT, status STRING")
display(validation_df)

invalid = [r for r in results if r[4] != "OK"]
if invalid:
    raise ValueError(f"Invalid test data, aborting inserts: {invalid}")

In [0]:
# Idempotent MERGE-inserts into bronze (insert-only: WHEN NOT MATCHED THEN INSERT)
# SQL is literal text with constant table names - no user-data interpolation.
insert_outcomes = []

def merge_insert(df, view, target_table, key_col, label):
    """MERGE test rows into a bronze table; records before/after counts as evidence."""
    df.createOrReplaceTempView(view)
    before = spark.table(target_table).count()
    spark.sql(f"""
        MERGE INTO {target_table} AS t
        USING {view} AS s
        ON t.{key_col} = s.{key_col}
        WHEN NOT MATCHED THEN INSERT *
    """)
    after = spark.table(target_table).count()
    insert_outcomes.append((label, target_table, before, after, after - before, "OK"))

try:
    merge_insert(test_airports,   "test_airports_src",   "claudecatalog.bronze.airports",   "airport_id",   "airports")
    merge_insert(test_passengers, "test_passengers_src", "claudecatalog.bronze.passengers", "passenger_id", "passengers")
    merge_insert(test_bookings,   "test_bookings_src",   "claudecatalog.bronze.bookings",   "booking_id",   "bookings")
except Exception as e:
    insert_outcomes.append(("merge", "n/a", None, None, None, f"INSERT_ERROR: {type(e).__name__}: {e}"))

outcomes_df = spark.createDataFrame(
    insert_outcomes,
    schema="entity STRING, target_table STRING, rows_before INT, rows_after INT, rows_inserted INT, status STRING")
display(outcomes_df)

errors = [o for o in insert_outcomes if o[5] != "OK"]
if errors:
    print("Bronze inserts FAILED:", errors)
else:
    print("Bronze inserts complete. Trigger a pipeline update, then run 02_validate_cdc_flow.")